In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

data = [(1, 'Eva', 2450), (2, 'Charlie', 3097), (3, 'Ivy', 1953), (4, 'Bob', 3924), (5, 'Bob', 4746), (6, 'Frank', 3120), (7, 'Frank', 1431), (8, 'Frank', 801), (9, 'Jack', 4388), (10, 'Eva', 3268), (11, 'Ivy', 2794), (12, 'Jack', 4153), (13, 'Alice', 4410), (14, 'Ivy', 2865), (15, 'Frank', 2337), (16, 'Grace', 2575), (17, 'Charlie', 2446), (18, 'Bob', 2462), (19, 'Frank', 682), (20, 'Charlie', 819), (21, 'Eva', 4654), (22, 'Jack', 3449), (23, 'Ivy', 4630), (24, 'Eva', 3654), (25, 'Bob', 2662), (26, 'Frank', 741), (27, 'Ivy', 3688), (28, 'Jack', 4729), (29, 'Alice', 2930), (30, 'Charlie', 1085), (31, 'David', 4096), (32, 'Helen', 523), (33, 'Jack', 868), (34, 'Eva', 3094), (35, 'Charlie', 4403), (36, 'Bob', 1953), (37, 'Frank', 1996), (38, 'Alice', 1930), (39, 'Grace', 4408), (40, 'Alice', 4145), (41, 'David', 2478), (42, 'Charlie', 3403), (43, 'David', 2553), (44, 'Grace', 2456), (45, 'Charlie', 3523), (46, 'Frank', 2434), (47, 'Charlie', 2844), (48, 'Jack', 3830), (49, 'Frank', 1889), (50, 'Grace', 3858), (51, 'Eva', 4345), (52, 'Alice', 1584), (53, 'Charlie', 1359), (54, 'Bob', 3538), (55, 'Charlie', 983), (56, 'Frank', 696), (57, 'Alice', 675), (58, 'Alice', 3153), (59, 'Alice', 1802), (60, 'Eva', 4376)]


In [0]:
df = spark.createDataFrame(data, ["id","name","amount"])

In [0]:

df.write.format('delta').mode('overwrite').save('/Volumes/workspace/default/sample/Delta_table1')

In [0]:
df=DeltaTable.forPath(spark,'/Volumes/workspace/default/sample/Delta_table1')

In [0]:
insert_data=[(1, "Alice", 3000),  
    (2, "Bob", 4000),    
    (61, "NewCust1", 2000),  
    (62, "NewCust2", 3500)   
]

In [0]:
ins_df=spark.createDataFrame(insert_data, ["id","name","amount"])

In [0]:
df.alias("target").merge(
    ins_df.alias('Source'),
    'target.id = Source.id'
).whenMatchedUpdate(
    set = {
        "name": "Source.name",
        "amount": "Source.amount"
    }
).whenNotMatchedInsert(values = {
    "id": "Source.id",
    "name": "Source.name",
    "amount": "Source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
updated_df = spark.read.format('delta').load('/Volumes/workspace/default/sample/Delta_table1')

In [0]:
display(updated_df)

id,name,amount,Category
3,Ivy,1953,null
4,Bob,3924,null
5,Bob,4746,null
6,Frank,3120,null
7,Frank,1431,null
8,Frank,801,null
9,Jack,4388,null
10,Eva,3268,null
11,Ivy,2794,null
12,Jack,4153,null


In [0]:
df.delete('amount<1000')

DataFrame[num_affected_rows: bigint]

In [0]:
display(df.toDF())

id,name,amount,Category
3,Ivy,1953,null
4,Bob,3924,null
5,Bob,4746,null
6,Frank,3120,null
7,Frank,1431,null
9,Jack,4388,null
10,Eva,3268,null
11,Ivy,2794,null
12,Jack,4153,null
13,Alice,4410,null


In [0]:
updated_df=updated_df.withColumn(
    'Category',
    when(col('amount')<1000,'Low')
    .when(col('amount')>=1000,'Medium')
    .when(col('amount')>=2000,'High')
    .when(col('amount')>=3000,'Very High'
).otherwise('Unknown')
)

Schema Evolution

In [0]:
updated_df.write.format('delta').option('overwriteSchema','true').mode('overwrite').save('/Volumes/workspace/default/sample/Delta_table1')

In [0]:
display(updated_df)


id,name,amount,Category
3,Ivy,1953,Medium
4,Bob,3924,Medium
5,Bob,4746,Medium
6,Frank,3120,Medium
7,Frank,1431,Medium
9,Jack,4388,Medium
10,Eva,3268,Medium
11,Ivy,2794,Medium
12,Jack,4153,Medium
13,Alice,4410,Medium


In [0]:
updated_df=DeltaTable.forPath(spark,'/Volumes/workspace/default/sample/Delta_table1')


In [0]:
updated_df.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
15,2026-04-19T13:55:46.000Z,71719280677188,bhavagnaprathipati@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(791015764488259),ad44e7c2-026f-4d48-8e11-1e50e1580767,0419-131642-96w44585-v2n,14,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1791, numDeletionVectorsRemoved -> 0, numOutputRows -> 53, numOutputBytes -> 1857)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-19T13:55:45.000Z,71719280677188,bhavagnaprathipati@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(791015764488259),49d5fd74-9ee2-4bea-8a04-8833c35d152f,0419-131642-96w44585-v2n,13,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 6438, p25FileSize -> 1791, numDeletionVectorsRemoved -> 1, minFileSize -> 1791, numAddedFiles -> 1, maxFileSize -> 1791, p75FileSize -> 1791, p50FileSize -> 1791, numAddedBytes -> 1791)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-19T13:55:43.000Z,71719280677188,bhavagnaprathipati@gmail.com,DELETE,"Map(predicate -> [""(amount#12962L < 1000)""])",null,List(791015764488259),49d5fd74-9ee2-4bea-8a04-8833c35d152f,0419-131642-96w44585-v2n,12,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1608, numDeletionVectorsUpdated -> 1, numDeletedRows -> 9, scanTimeMs -> 1011, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 597)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-19T13:55:38.000Z,71719280677188,bhavagnaprathipati@gmail.com,MERGE,"Map(predicate -> [""(id#12634L = id#12638L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(791015764488259),9dd99321-882f-47f2-973e-e26373cf80f0,0419-131642-96w44585-v2n,11,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 4763, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3779, materializeSourceTimeMs -> 121, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1508, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2063)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-19T13:55:31.000Z,71719280677188,bhavagnaprathipati@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(791015764488259),ed6a6a8c-82d4-4ebd-9e32-786a175134d4,0419-131642-96w44585-v2n,10,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1857, numDeletionVectorsRemoved -> 0, numOutputRows -> 60, numOutputBytes -> 1675)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-19T13:55:12.000Z,71719280677188,bhavagnaprathipati@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(791015764488259),08b18e4f-5c2b-4b5e-85f6-bbbbc76d4b23,0419-131642-96w44585-v2n,9,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1630, numDeletionVectorsRemoved -> 0, numOutputRows -> 53, numOutputBytes -> 1857)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-19T13:39:02.000Z,71719280677188,bhavagnaprathipati@gmail.com,OPTIMIZE,"Map(pre

In [0]:
updated_df.restoreToVersion(6)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [0]:
d1=spark.read.format('delta').load('/Volumes/workspace/default/sample/Delta_table1')
display(d1)


id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268
